# BlueBikes Visualizations
## How Did the Pandemic Reshape Boston's Bike-Share?

This notebook generates all 5 visualizations for our DS4200 project. Each reads from a pre-aggregated dataset computed from the full ~9.9 million trips.

> **Overview** (rider mix shift) → **When** (temporal patterns) → **Who** (demographics) → **Where** (station geography) → **How** (ride behavior)

## Imports

In [1]:
import pandas as pd
import altair as alt
import os

---
## Visualization 1: Monthly Ridership Over Time

**Question:** How has overall ridership changed post-pandemic?

This line chart shows total trips per month across all three years. Clicking a year in the legend isolates it. Hovering any point reveals the subscriber/casual breakdown for that month.

### Load Data

In [2]:
viz1 = pd.read_csv('data/viz1_ridership.csv')

COLORS = {
    'Pre-Pandemic (2019)':  '#BA7517',
    'Post-Pandemic (2022)': '#5DCAA5',
    'Post-Pandemic (2023)': '#085041',
}
PERIOD_ORDER = ['Pre-Pandemic (2019)', 'Post-Pandemic (2022)', 'Post-Pandemic (2023)']
MONTH_ORDER  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = (
    viz1.groupby(['period','month','month_name'], as_index=False)
        .agg(month_total=('trip_count','sum'))
)
sub = (
    viz1[viz1['usertype'] == 'Subscriber']
        [['period','month','month_name','trip_count','avg_duration_min']]
        .rename(columns={'trip_count':'sub_count','avg_duration_min':'sub_avg_dur'})
)
monthly = monthly.merge(sub, on=['period','month','month_name'], how='left')
monthly['sub_pct']   = (monthly['sub_count'] / monthly['month_total'] * 100).round(1)
monthly['cus_pct']   = (100 - monthly['sub_pct']).round(1)
monthly['cus_count'] = monthly['month_total'] - monthly['sub_count']

monthly.head()

,period,month,month_name,month_total,sub_count,sub_avg_dur,sub_pct,cus_pct,cus_count
0,Post-Pandemic (2022),1,Jan,81478,72611,12.3,89.1,10.9,8867
1,Post-Pandemic (2022),2,Feb,110248,94843,12.4,86.0,14.0,15405
2,Post-Pandemic (2022),3,Mar,181849,142666,12.7,78.5,21.5,39183
3,Post-Pandemic (2022),4,Apr,273170,199985,12.9,73.2,26.8,73185
4,Post-Pandemic (2022),5,May,349030,234320,13.8,67.1,32.9,114710


### Chart

In [3]:
year_select = alt.selection_point(fields=['period'], bind='legend')

tooltip = [
    alt.Tooltip('month_name:N',  title='Month'),
    alt.Tooltip('month_total:Q', title='Total trips',    format=','),
    alt.Tooltip('sub_count:Q',   title='Subscribers',    format=','),
    alt.Tooltip('sub_pct:Q',     title='Subscriber %',   format='.1f'),
    alt.Tooltip('cus_count:Q',   title='Casual riders',  format=','),
    alt.Tooltip('cus_pct:Q',     title='Casual %',       format='.1f'),
]

base = alt.Chart(monthly).encode(
    x=alt.X('month_name:N',
            sort=MONTH_ORDER,
            title='Month',
            axis=alt.Axis(labelAngle=0)),
    color=alt.Color('period:N',
                    sort=PERIOD_ORDER,
                    scale=alt.Scale(
                        domain=list(COLORS.keys()),
                        range=list(COLORS.values())
                    ),
                    legend=alt.Legend(
                        title=None,
                        orient='top',
                        symbolType='stroke',
                        symbolStrokeWidth=3,
                        labelFontSize=12,
                    )),
)

lines = base.mark_line(
    strokeWidth=2.5,
    point=alt.OverlayMarkDef(size=50, filled=True)
).encode(
    y=alt.Y('month_total:Q',
            title='Total Trips',
            axis=alt.Axis(format='~s'),
            scale=alt.Scale(domainMin=0)),
    opacity=alt.condition(year_select, alt.value(1.0), alt.value(0.15)),
    tooltip=tooltip
).add_params(year_select)

viz1_chart = lines.properties(
    title='Monthly Ridership: 2019, 2022, and 2023',
    width=620,
    height=340,
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True,
    gridColor='#eeeeee',
    labelFontSize=11,
    titleFontSize=12,
).configure_legend(
    labelFontSize=12,
    symbolSize=120,
    padding=8,
)

viz1_chart.display()

alt.Chart(...)

---
## Visualization 2: Activity Heatmap

**Question:** When do people ride now vs before the pandemic?

Three heatmaps stacked vertically show average trips per hour per day of week, one for each year. All share the same color scale so the panels are directly comparable. The commute peaks in 2019 visibly flatten in 2022 and 2023.

### Load Data

In [4]:
viz2 = pd.read_csv('data/viz2_heatmap.csv')

PERIOD_LABELS = {
    'Pre-Pandemic (2019)':  '2019 - Pre-Pandemic',
    'Post-Pandemic (2022)': '2022 - Recovery',
    'Post-Pandemic (2023)': '2023 - Post-Pandemic',
}
viz2['year_label'] = viz2['period'].map(PERIOD_LABELS)

viz2.head()

,year,day_of_week,hour,total_trips,num_days,avg_trips,period,year_label
0,2019,Friday,0,2899,52,55.8,Pre-Pandemic (2019),2019 - Pre-Pandemic
1,2019,Friday,1,1590,52,30.6,Pre-Pandemic (2019),2019 - Pre-Pandemic
2,2019,Friday,2,974,52,18.7,Pre-Pandemic (2019),2019 - Pre-Pandemic
3,2019,Friday,3,388,52,7.5,Pre-Pandemic (2019),2019 - Pre-Pandemic
4,2019,Friday,4,485,52,9.3,Pre-Pandemic (2019),2019 - Pre-Pandemic


### Chart

In [5]:
DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
LABEL_ORDER = [PERIOD_LABELS[p] for p in ['Pre-Pandemic (2019)', 'Post-Pandemic (2022)', 'Post-Pandemic (2023)']]

global_max = viz2['avg_trips'].max()

base = alt.Chart(viz2).mark_rect(
    cornerRadius=2
).encode(
    x=alt.X('hour:O',
            title='Hour of Day',
            axis=alt.Axis(
                labelAngle=0,
                values=list(range(0, 24, 3)),
                labelExpr=(
                    "datum.value === 0 ? '12a' : "
                    "datum.value < 12 ? datum.value + 'a' : "
                    "datum.value === 12 ? '12p' : "
                    "(datum.value - 12) + 'p'"
                )
            )),
    y=alt.Y('day_of_week:O',
            sort=DAY_ORDER,
            title=None,
            axis=alt.Axis(labelFontSize=12)),
    color=alt.Color('avg_trips:Q',
                    title='Avg Trips',
                    scale=alt.Scale(
                        domainMin=0,
                        domainMax=global_max,
                        scheme='tealblues'
                    ),
                    legend=alt.Legend(
                        orient='right',
                        gradientLength=180,
                        gradientThickness=12,
                        titleFontSize=11,
                        labelFontSize=10,
                    )),
    tooltip=[
        alt.Tooltip('day_of_week:N', title='Day'),
        alt.Tooltip('hour:O',        title='Hour'),
        alt.Tooltip('avg_trips:Q',   title='Avg Trips',   format='.0f'),
        alt.Tooltip('total_trips:Q', title='Total Trips',  format=','),
    ]
)

viz2_chart = base.facet(
    row=alt.Row(
        'year_label:N',
        sort=LABEL_ORDER,
        title=None,
        header=alt.Header(
            labelFontSize=14,
            labelFontWeight='normal',
            labelColor='#444',
            labelPadding=10,
        )
    )
).resolve_scale(
    color='shared'
).properties(
    title=alt.TitleParams(
        text='Ride Activity by Hour and Day of Week',
        anchor='middle',
        offset=15,
    )
).configure_view(
    stroke=None,
).configure_axis(
    grid=False,
    labelFontSize=11,
    titleFontSize=11,
).configure_facet(
    spacing=12
)

viz2_chart.display()

alt.FacetChart(...)

---
## Visualization 3: Population Pyramid

**Question:** Who is riding? (2019 data only)

Age and gender distribution of riders. Male bars extend left, female bars extend right, both showing positive percentages.

### Load Data

In [6]:
viz3 = pd.read_csv('data/viz3_demographics.csv')
viz3.head()

,age_group,gender_label,count,percentage,day_type,plot_pct
0,15-19,Female,7868,0.45,Weekday,0.45
1,15-19,Male,27912,1.59,Weekday,-1.59
2,20-24,Female,92368,5.26,Weekday,5.26
3,20-24,Male,237058,13.50,Weekday,-13.50
4,25-29,Female,132267,7.53,Weekday,7.53


### Chart

In [7]:
day_type_select = alt.selection_point(
    fields=['day_type'],
    bind=alt.binding_radio(
        options=['Weekday', 'Weekend', 'Total'],
        name='Day Type: '
    ),
    value='Total'
)

viz3_chart = alt.Chart(viz3).mark_bar().encode(
    y=alt.Y('age_group:O',
            title='Age Group',
            sort=alt.SortOrder('descending')),
    x=alt.X('plot_pct:Q',
            title='Percentage of Riders',
            axis=alt.Axis(
                format='.1f',
                labelExpr="abs(datum.value) + '%'"
            ),
            scale=alt.Scale(domain=[-25, 15])),
    color=alt.Color('gender_label:N',
                    title='Gender',
                    scale=alt.Scale(
                        domain=['Male', 'Female'],
                        range=['#2171b5', '#e377c2']
                    ),
                    legend=alt.Legend(orient='bottom')),
    tooltip=[
        alt.Tooltip('age_group:N', title='Age Group'),
        alt.Tooltip('gender_label:N', title='Gender'),
        alt.Tooltip('percentage:Q', title='%', format='.1f'),
        alt.Tooltip('count:Q', title='Riders', format=',')
    ]
).add_params(
    day_type_select
).transform_filter(
    day_type_select
).properties(
    title='2019 Rider Demographics: Age and Gender Distribution',
    width=600,
    height=400
)

viz3_chart.display()

alt.Chart(...)

---
## Visualization 4: Station Map (D3.js)

**Question:** Where do people ride, and how has the network grown?

The station map is built with D3.js and embedded directly in the website. The dataset is already prepared by the pipeline (`viz4_stations.csv`).

In [8]:
viz4 = pd.read_csv('data/viz4_stations.csv')
print(f"Stations per year:")
print(viz4.groupby('year')['station_id'].nunique().to_string())

Stations per year:
year
2019    332
2022    440
2023    805


---
## Visualization 5: Trip Duration Distribution

**Question:** How has riding behavior changed?

This two-panel chart shows trip duration distributions across all three years. The top line chart has a brush — drag to select a range and the bottom bar chart zooms into that range. A rider type filter lets you compare subscribers vs casual riders.

### Load Data

In [9]:
viz5 = pd.read_csv('data/viz5_duration.csv')

viz5_agg = (
    viz5.groupby(['period', 'bin_start', 'usertype'], as_index=False)
        .agg(count=('count', 'sum'))
)

viz5_both = (
    viz5_agg.groupby(['period', 'bin_start'], as_index=False)
            .agg(count=('count', 'sum'))
)
viz5_both['usertype'] = 'Both'
viz5_combined = pd.concat([viz5_agg, viz5_both], ignore_index=True)

viz5_combined.head()

,period,bin_start,usertype,count
0,Post-Pandemic (2022),0,Customer,2510
1,Post-Pandemic (2022),0,Subscriber,26043
2,Post-Pandemic (2022),2,Customer,12664
3,Post-Pandemic (2022),2,Subscriber,199308
4,Post-Pandemic (2022),4,Customer,36221


### Chart

In [10]:
PERIOD_ORDER = ['Pre-Pandemic (2019)', 'Post-Pandemic (2022)', 'Post-Pandemic (2023)']
PERIOD_COLORS = {
    'Pre-Pandemic (2019)':  '#BA7517',
    'Post-Pandemic (2022)': '#5DCAA5',
    'Post-Pandemic (2023)': '#085041',
}

usertype_select = alt.selection_point(
    fields=['usertype'],
    bind=alt.binding_radio(
        options=['Both', 'Subscriber', 'Customer'],
        labels=['Both', 'Subscriber', 'Customer'],
        name='Rider type:  '
    ),
    value='Both'
)

brush = alt.selection_interval(encodings=['x'])

color = alt.Color('period:N',
                  sort=PERIOD_ORDER,
                  scale=alt.Scale(
                      domain=list(PERIOD_COLORS.keys()),
                      range=list(PERIOD_COLORS.values())
                  ),
                  legend=alt.Legend(
                      title=None,
                      orient='top',
                      labelFontSize=11,
                      symbolType='square',
                      symbolSize=120,
                  ))

viz5_tooltip = [
    alt.Tooltip('period:N',    title='Year'),
    alt.Tooltip('bin_start:Q', title='Duration (min)'),
    alt.Tooltip('count:Q',     title='Trips', format=','),
    alt.Tooltip('usertype:N',  title='Rider type'),
]

overview = alt.Chart(viz5_combined).mark_line(
    opacity=0.85,
    strokeWidth=2,
    point=alt.OverlayMarkDef(size=30, filled=True)
).encode(
    x=alt.X('bin_start:Q',
            title='Trip duration (minutes)',
            scale=alt.Scale(domainMin=0),
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q',
            title='Number of trips',
            stack=None),
    color=color,
    tooltip=viz5_tooltip,
).add_params(
    brush,
    usertype_select,
).transform_filter(
    usertype_select
).properties(
    title='Trip Duration Distribution',
    width=580,
    height=220,
)

detail = alt.Chart(viz5_combined).mark_bar(
    opacity=0.85,
).encode(
    x=alt.X('bin_start:O',
            title='Trip duration (minutes)',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q',
            title='Number of trips',
            stack=None),
    color=alt.Color('period:N',
                    sort=PERIOD_ORDER,
                    scale=alt.Scale(
                        domain=list(PERIOD_COLORS.keys()),
                        range=list(PERIOD_COLORS.values())
                    ),
                    legend=None),
    xOffset=alt.XOffset('period:N', sort=PERIOD_ORDER),
    tooltip=viz5_tooltip,
).transform_filter(
    brush
).transform_filter(
    usertype_select
).properties(
    title='Zoomed View',
    width=580,
    height=220,
)

viz5_chart = (overview & detail).configure_view(
    strokeWidth=0,
).configure_axis(
    grid=True,
    gridColor='#eeeeee',
    labelFontSize=11,
    titleFontSize=12,
)

viz5_chart.display()

alt.VConcatChart(...)

---
## Export Visualizations as HTML

In [11]:
os.makedirs('viz_html', exist_ok=True)

viz1_chart.save('viz_html/viz1_rider_mix.html')
viz2_chart.save('viz_html/viz2_heatmap.html')
viz3_chart.save('viz_html/viz3_pyramid.html')
viz5_chart.save('viz_html/viz5_duration.html')

print('Saved all Altair charts to viz_html/')

Saved all Altair charts to viz_html/
